# DuckLake Streaming Demo (Jupyter)

Mirrors `streaming_demo.py` (marimo). Generate synthetic data with a parametrized row
count, write it to DuckLake via streaming, and query it interactively. Setup and write
are Python; queries use the engine's DuckDB connection directly via `con.execute(...).pl()`.


In [ ]:
import sys
from pathlib import Path

from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

_REPO = Path.cwd()
if _REPO.name == "notebooks":
    _REPO = _REPO.parent
if str(_REPO / "src") not in sys.path:
    sys.path.insert(0, str(_REPO / "src"))

from ducklake_playground import (
    DuckLakeEngine,
    GeneratorSpec,
    StreamingGenerator,
    load_config,
    measure_time_and_memory,
)

config = load_config(_REPO / "config.yaml")
config.name

## Parameters


In [ ]:
ROW_COUNT = 1_000_000
TABLE_NAME = "demo_table"
STORAGE_MODE = "local"
CHUNK_SIZE = config.batch_size
SEED = config.schema.seed

assert STORAGE_MODE in config.storage_modes
f"Will create {TABLE_NAME} with {ROW_COUNT:,} rows in {STORAGE_MODE} storage"

In [ ]:
engine = DuckLakeEngine()
engine.setup(config, STORAGE_MODE)
con = engine.connection
catalog = engine.catalog_name
fq = f"{catalog}.main.{TABLE_NAME}"
fq

## Write-time options

Persisted in the Postgres catalog. Set BEFORE writing.


In [ ]:
con.execute(f"CALL {catalog}.set_option('parquet_version', 2)")
con.execute(f"CALL {catalog}.set_option('parquet_compression', 'zstd')")
con.execute(f"CALL {catalog}.set_option('parquet_row_group_size_bytes', '16MB')")
"options set"

## Stream-generate + write


In [ ]:
gen = StreamingGenerator(
    GeneratorSpec(
        schema_config=config.schema,
        total_rows=ROW_COUNT,
        chunk_size=CHUNK_SIZE,
        seed=SEED,
    )
)
with measure_time_and_memory() as t:
    engine.write_overwrite(TABLE_NAME, gen.arrow_reader(), gen.schema)
timing = t[0]
print(
    f"Wrote {ROW_COUNT:,} rows in {timing.wall_time_seconds:.2f}s | "
    f"peak RSS {timing.peak_rss_mb:.0f} MB (delta {timing.delta_rss_mb:.0f} MB)"
)

## Schema + sanity counts


In [ ]:
con.execute(f"DESCRIBE {fq}").pl()

In [ ]:
con.execute(
    f"""
    SELECT COUNT(*) AS n,
           MIN(event_date) AS min_d,
           MAX(event_date) AS max_d,
           COUNT(DISTINCT event_date) AS distinct_partitions
    FROM {fq}
    """
).pl()

## Query


In [ ]:
query = f"""
SELECT varchar_col,
       COUNT(*) AS cnt,
       SUM(int64_col) AS sum_val,
       AVG(float64_col) AS avg_val,
       MIN(event_date) AS min_date,
       MAX(event_date) AS max_date
FROM {fq}
WHERE event_date BETWEEN DATE '2024-01-10' AND DATE '2024-01-20'
GROUP BY varchar_col
ORDER BY cnt DESC
LIMIT 20
"""
with measure_time_and_memory() as t:
    result = con.execute(query).pl()
timing = t[0]
print(
    f"{result.height:,} rows in {timing.wall_time_seconds:.3f}s | "
    f"peak RSS {timing.peak_rss_mb:.0f} MB (delta {timing.delta_rss_mb:.0f} MB)"
)
result

## Snapshots + maintenance


In [ ]:
con.execute(f"SELECT * FROM {catalog}.snapshots() ORDER BY snapshot_id DESC LIMIT 10").pl()

In [ ]:
# Run after several writes to compact files. Order matters.
# con.execute(f"CALL ducklake_merge_adjacent_files('{catalog}')")
# con.execute(f"CALL ducklake_expire_snapshots('{catalog}', older_than => INTERVAL '7 days')")
# con.execute(f"CALL ducklake_cleanup_old_files('{catalog}', cleanup_all => true)")

In [ ]:
# Drop the demo table when finished. Comment out to keep iterating.
# engine.teardown(TABLE_NAME)
# engine.close()